# Kansas State HMDA Data Combiner

This notebook reads the individual Kansas state HMDA data files for 2022, 2023, and 2024, combines them into a single dataset, and saves the merged data as `state_KS.csv`.

## Objective
- Load data from 2022_state_KS.csv, 2023_state_KS.csv, and 2024_state_KS.csv
- Combine all records into a single comprehensive dataset
- Perform basic data validation and quality checks
- Save the combined dataset as state_KS.csv

## 1. Import Required Libraries

Import necessary libraries for data processing and file operations.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', None)

print("✅ Libraries imported successfully!")
print(f"📅 Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Current date: 2025-10-06 17:28:28


## 2. Define File Paths

Set up the paths for the input files and output file.

In [2]:
# Define data directory and file paths
data_dir = Path("../data/actual")

# Input files
input_files = {
    "2022": data_dir / "2022_state_KS.csv",
    "2023": data_dir / "2023_state_KS.csv", 
    "2024": data_dir / "2024_state_KS.csv"
}

# Output file
output_file = data_dir / "state_KS.csv"

print("📁 File paths configured:")
print(f"   Data directory: {data_dir}")
print(f"   Output file: {output_file}")
print("\n📂 Input files:")
for year, filepath in input_files.items():
    exists = "✅" if filepath.exists() else "❌"
    print(f"   {year}: {filepath} {exists}")

📁 File paths configured:
   Data directory: ../data/actual
   Output file: ../data/actual/state_KS.csv

📂 Input files:
   2022: ../data/actual/2022_state_KS.csv ✅
   2023: ../data/actual/2023_state_KS.csv ✅
   2024: ../data/actual/2024_state_KS.csv ✅


## 3. Load Individual Data Files

Read each CSV file and examine the structure and content.

In [3]:
# Load each data file
dataframes = {}
total_records = 0

print("📊 Loading Kansas state HMDA data files...")
print("=" * 50)

for year, filepath in input_files.items():
    if filepath.exists():
        print(f"\n📂 Loading {year} data from: {filepath.name}")
        
        try:
            # Load the CSV file
            df = pd.read_csv(filepath)
            dataframes[year] = df
            
            # Display basic information
            print(f"   ✅ Loaded successfully")
            print(f"   📊 Shape: {df.shape}")
            print(f"   📅 Activity years in data: {sorted(df['activity_year'].unique()) if 'activity_year' in df.columns else 'No activity_year column'}")
            
            total_records += len(df)
            
            # Show first few columns to verify structure
            print(f"   🔍 First 5 column names: {list(df.columns[:5])}")
            
        except Exception as e:
            print(f"   ❌ Error loading {year} data: {str(e)}")
    else:
        print(f"\n❌ File not found: {filepath}")

print(f"\n📈 Summary:")
print(f"   Files loaded: {len(dataframes)}")
print(f"   Total records: {total_records:,}")

📊 Loading Kansas state HMDA data files...

📂 Loading 2022 data from: 2022_state_KS.csv
   ✅ Loaded successfully
   📊 Shape: (109754, 99)
   📅 Activity years in data: [np.int64(2022)]
   🔍 First 5 column names: ['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code']

📂 Loading 2023 data from: 2023_state_KS.csv
   ✅ Loaded successfully
   📊 Shape: (109754, 99)
   📅 Activity years in data: [np.int64(2022)]
   🔍 First 5 column names: ['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code']

📂 Loading 2023 data from: 2023_state_KS.csv
   ✅ Loaded successfully
   📊 Shape: (85830, 99)
   📅 Activity years in data: [np.int64(2023)]
   🔍 First 5 column names: ['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code']

📂 Loading 2024 data from: 2024_state_KS.csv
   ✅ Loaded successfully
   📊 Shape: (85830, 99)
   📅 Activity years in data: [np.int64(2023)]
   🔍 First 5 column names: ['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code']


## 4. Examine Data Structure and Consistency

Check that all files have the same column structure before combining.

In [4]:
# Check column consistency across files
print("🔍 COLUMN STRUCTURE ANALYSIS")
print("=" * 30)

if len(dataframes) > 0:
    # Get columns from first dataframe as reference
    reference_year = list(dataframes.keys())[0]
    reference_columns = set(dataframes[reference_year].columns)
    
    print(f"📋 Reference columns from {reference_year}: {len(reference_columns)} columns")
    
    # Check each dataframe against reference
    columns_consistent = True
    
    for year, df in dataframes.items():
        current_columns = set(df.columns)
        
        if current_columns == reference_columns:
            print(f"   ✅ {year}: Column structure matches ({len(current_columns)} columns)")
        else:
            print(f"   ⚠️ {year}: Column structure differs")
            
            missing = reference_columns - current_columns
            extra = current_columns - reference_columns
            
            if missing:
                print(f"      Missing columns: {missing}")
            if extra:
                print(f"      Extra columns: {extra}")
            
            columns_consistent = False
    
    if columns_consistent:
        print("\n✅ All files have consistent column structure - ready to combine!")
    else:
        print("\n⚠️ Column structure inconsistencies detected - will use intersection of columns")
        
        # Find common columns
        common_columns = reference_columns
        for year, df in dataframes.items():
            common_columns = common_columns.intersection(set(df.columns))
        
        print(f"📊 Common columns across all files: {len(common_columns)}")
        
        # Update dataframes to use only common columns
        for year in dataframes.keys():
            dataframes[year] = dataframes[year][sorted(common_columns)]
            
else:
    print("❌ No data files loaded successfully")

🔍 COLUMN STRUCTURE ANALYSIS
📋 Reference columns from 2022: 99 columns
   ✅ 2022: Column structure matches (99 columns)
   ✅ 2023: Column structure matches (99 columns)
   ✅ 2024: Column structure matches (99 columns)

✅ All files have consistent column structure - ready to combine!


## 5. Combine All Data Files

Merge all the individual year files into a single comprehensive dataset.

In [5]:
# Combine all dataframes
if len(dataframes) > 0:
    print("🔄 Combining Kansas state HMDA data...")
    print("=" * 40)
    
    # Concatenate all dataframes
    combined_df = pd.concat(dataframes.values(), ignore_index=True)
    
    print(f"✅ Successfully combined {len(dataframes)} files")
    print(f"📊 Combined dataset shape: {combined_df.shape}")
    print(f"📅 Activity years in combined data: {sorted(combined_df['activity_year'].unique()) if 'activity_year' in combined_df.columns else 'No activity_year column'}")
    
    # Show breakdown by year
    if 'activity_year' in combined_df.columns:
        print("\n📈 Records by year:")
        year_counts = combined_df['activity_year'].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   {year}: {count:,} records")
    
    # Show basic statistics
    print(f"\n📋 Data Quality Summary:")
    print(f"   Total records: {len(combined_df):,}")
    print(f"   Total columns: {len(combined_df.columns)}")
    print(f"   Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Check for duplicates
    duplicate_count = combined_df.duplicated().sum()
    print(f"   Duplicate rows: {duplicate_count:,}")
    
    if duplicate_count > 0:
        print(f"   🔧 Removing {duplicate_count:,} duplicate rows...")
        combined_df = combined_df.drop_duplicates()
        print(f"   ✅ Final dataset shape: {combined_df.shape}")
        
else:
    print("❌ No data to combine - no files were loaded successfully")
    combined_df = None

🔄 Combining Kansas state HMDA data...
✅ Successfully combined 3 files
📊 Combined dataset shape: (290114, 99)
📅 Activity years in combined data: [np.int64(2022), np.int64(2023), np.int64(2024)]

📈 Records by year:
   2022: 109,754 records
   2023: 85,830 records
   2024: 94,530 records

📋 Data Quality Summary:
   Total records: 290,114
   Total columns: 99
   Memory usage: 531.9 MB
   Memory usage: 531.9 MB
   Duplicate rows: 945
   🔧 Removing 945 duplicate rows...
   Duplicate rows: 945
   🔧 Removing 945 duplicate rows...
   ✅ Final dataset shape: (289169, 99)
   ✅ Final dataset shape: (289169, 99)


## 6. Data Quality Validation

Perform basic validation checks on the combined dataset.

In [6]:
# Perform data quality checks
if combined_df is not None:
    print("🔍 DATA QUALITY VALIDATION")
    print("=" * 30)
    
    # Display first few rows
    print("📋 Sample of combined data:")
    print(combined_df.head())
    
    print("\n📊 Data Info:")
    print(combined_df.info())
    
    # Check for missing values in key columns
    print("\n🔍 Missing Values Analysis:")
    key_columns = ['activity_year', 'state_code', 'county_code', 'action_taken']
    available_key_columns = [col for col in key_columns if col in combined_df.columns]
    
    if available_key_columns:
        missing_summary = combined_df[available_key_columns].isnull().sum()
        print(missing_summary)
        
        # Check if all records are for Kansas (state_code should be 'KS')
        if 'state_code' in combined_df.columns:
            state_counts = combined_df['state_code'].value_counts()
            print(f"\n🗺️ State distribution:")
            print(state_counts)
            
            if len(state_counts) == 1 and state_counts.index[0] == 'KS':
                print("   ✅ All records are for Kansas state")
            else:
                print("   ⚠️ Found records for states other than Kansas")
    
    # Basic statistics for numeric columns
    numeric_columns = combined_df.select_dtypes(include=[np.number]).columns
    if len(numeric_columns) > 0:
        print(f"\n📈 Found {len(numeric_columns)} numeric columns")
        print("Sample statistics for key numeric fields:")
        
        # Show stats for some key fields if they exist
        key_numeric = ['loan_amount', 'income', 'property_value']
        available_numeric = [col for col in key_numeric if col in combined_df.columns]
        
        if available_numeric:
            print(combined_df[available_numeric].describe())
    
    print("\n✅ Data quality validation completed")
    
else:
    print("❌ No combined data available for validation")

🔍 DATA QUALITY VALIDATION
📋 Sample of combined data:
   activity_year                   lei  derived_msa-md state_code  \
0           2022  549300FGXN1K3HLB1R50           99999         KS   
1           2022  549300FGXN1K3HLB1R50           28140         KS   
2           2022  549300FGXN1K3HLB1R50           99999         KS   
3           2022  549300FGXN1K3HLB1R50           99999         KS   
4           2022  549300FGXN1K3HLB1R50           99999         KS   

   county_code  census_tract conforming_loan_limit derived_loan_product_type  \
0      20099.0  2.009995e+10                     C            FHA:First Lien   
1      20091.0  2.009105e+10                     C   Conventional:First Lien   
2      20111.0  2.011100e+10                     C   Conventional:First Lien   
3      20133.0  2.013395e+10                     C   Conventional:First Lien   
4      20113.0  2.011379e+10                     C   Conventional:First Lien   

              derived_dwelling_category        deri

## 7. Save Combined Dataset

Export the combined dataset to the output file.

In [7]:
# Save the combined dataset
if combined_df is not None:
    print("💾 SAVING COMBINED DATASET")
    print("=" * 30)
    
    try:
        # Save to CSV
        print(f"📁 Saving combined data to: {output_file}")
        combined_df.to_csv(output_file, index=False)
        
        # Verify the saved file
        file_size = output_file.stat().st_size / 1024**2  # Size in MB
        
        print(f"✅ File saved successfully!")
        print(f"📊 Records saved: {len(combined_df):,}")
        print(f"📁 File size: {file_size:.1f} MB")
        print(f"📅 Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Quick verification by reading back a few rows
        verification_df = pd.read_csv(output_file, nrows=5)
        print(f"\n🔍 Verification - first 5 rows of saved file:")
        print(verification_df)
        
    except Exception as e:
        print(f"❌ Error saving file: {str(e)}")
        
else:
    print("❌ No data to save")

💾 SAVING COMBINED DATASET
📁 Saving combined data to: ../data/actual/state_KS.csv
✅ File saved successfully!
📊 Records saved: 289,169
📁 File size: 105.2 MB
📅 Created: 2025-10-06 17:28:35

🔍 Verification - first 5 rows of saved file:
   activity_year                   lei  derived_msa-md state_code  \
0           2022  549300FGXN1K3HLB1R50           99999         KS   
1           2022  549300FGXN1K3HLB1R50           28140         KS   
2           2022  549300FGXN1K3HLB1R50           99999         KS   
3           2022  549300FGXN1K3HLB1R50           99999         KS   
4           2022  549300FGXN1K3HLB1R50           99999         KS   

   county_code  census_tract conforming_loan_limit derived_loan_product_type  \
0      20099.0  2.009995e+10                     C            FHA:First Lien   
1      20091.0  2.009105e+10                     C   Conventional:First Lien   
2      20111.0  2.011100e+10                     C   Conventional:First Lien   
3      20133.0  2.013395e+10     

## 8. Summary Report

Generate a comprehensive summary of the data combination process.

In [8]:
# Generate summary report
print("📋 KANSAS STATE DATA COMBINATION SUMMARY")
print("=" * 45)

if combined_df is not None:
    # Create detailed summary
    summary_info = {
        "Process Date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "Input Files": len(input_files),
        "Files Successfully Loaded": len(dataframes),
        "Total Records Combined": len(combined_df),
        "Total Columns": len(combined_df.columns),
        "Output File": str(output_file),
        "File Size (MB)": f"{output_file.stat().st_size / 1024**2:.1f}"
    }
    
    for key, value in summary_info.items():
        print(f"📊 {key}: {value}")
    
    # Year breakdown if available
    if 'activity_year' in combined_df.columns:
        print("\n📅 Records by Activity Year:")
        year_breakdown = combined_df['activity_year'].value_counts().sort_index()
        for year, count in year_breakdown.items():
            print(f"   {year}: {count:,} records")
    
    print(f"\n✅ SUCCESS: Kansas state HMDA data successfully combined!")
    print(f"📁 Combined dataset saved as: {output_file.name}")
    print(f"📊 Ready for analysis with {len(combined_df):,} loan records")
    
    print("\n🎯 Next Steps:")
    print("   1. Use state_KS.csv for comprehensive Kansas lending analysis")
    print("   2. Extract census tracts for synthetic data generation")
    print("   3. Analyze lending patterns across multiple years")
    print("   4. Train machine learning models on historical data")
    
else:
    print("❌ FAILED: No data could be combined")
    print("🔧 Please check:")
    print("   1. File paths are correct")
    print("   2. CSV files exist and are readable")
    print("   3. Files have compatible structure")

print("\n" + "=" * 45)
print("📋 PROCESS COMPLETE")
print("=" * 45)

📋 KANSAS STATE DATA COMBINATION SUMMARY
📊 Process Date: 2025-10-06 17:28:35
📊 Input Files: 3
📊 Files Successfully Loaded: 3
📊 Total Records Combined: 289169
📊 Total Columns: 99
📊 Output File: ../data/actual/state_KS.csv
📊 File Size (MB): 105.2

📅 Records by Activity Year:
   2022: 109,516 records
   2023: 85,653 records
   2024: 94,000 records

✅ SUCCESS: Kansas state HMDA data successfully combined!
📁 Combined dataset saved as: state_KS.csv
📊 Ready for analysis with 289,169 loan records

🎯 Next Steps:
   1. Use state_KS.csv for comprehensive Kansas lending analysis
   2. Extract census tracts for synthetic data generation
   3. Analyze lending patterns across multiple years
   4. Train machine learning models on historical data

📋 PROCESS COMPLETE
